In [40]:
import io
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from itertools import count
import pickle
import json
import boto3
import pg8000
from botocore.exceptions import ClientError
from pg8000.dbapi import DatabaseError, ProgrammingError
from dotenv import load_dotenv
from datetime import datetime, timedelta, time
import gc
import os
import hashlib

# --- S3 SETUP ---
BUCKET_NAME = "dsa3101-storm-tracking-tw08"

# create s3 client (this reads credentials from ~/.aws/credentials)
s3 = boto3.client("s3")


# helpers

In [41]:
####################################################################################################
# helpers 
# def get_secret(secret_arn):
#     resp = secrets_client.get_secret_value(SecretId=secret_arn)
#     return json.loads(resp['SecretString'])

# def get_db_conn(secret_arn):
#     global _db_conn
#     if _db_conn:
#         try:
#             cur = _db_conn.cursor()
#             cur.execute("SELECT 1;")
#             cur.close()
#             return _db_conn
#         except Exception:
#             _db_conn = None
#     secret = get_secret(secret_arn)
#     host = secret['host']
#     dbname = secret['dbname']
#     user = secret['username']
#     password = secret['password']
#     port = int(secret.get('port', 5432))
#     _db_conn = pg8000.connect(
#         host=host,
#         database=dbname,
#         user=user,
#         password=password,
#         port=port
#     )
#     return _db_conn

def query_to_df(conn, query, params=None):
    """
    Execute an SQL query and return results as a pandas DataFrame.

    Args:
        conn : active pg8000 connection
        query : SQL query string (use %s placeholders)
        params : tuple/list of parameters (optional)
    """
    with conn.cursor() as cur:
        cur.execute(query, params or ())
        # Extract column names from cursor description
        columns = [desc[0] for desc in cur.description]
        data = cur.fetchall()
    # Create DataFrame
    return pd.DataFrame(data, columns=columns)



####################################################################################################

##############################
# Matched labeled storms
# Using overlap and centroid distance between 2 consecutive frames
##############################
def match_frames(labels_t1, df_t1, labels_t2, df_t2, 
                 overlap_threshold=0.02, dist_threshold=20,
                 verbose=False):

    matches = []

    for _, row1 in df_t1.iterrows():
        id1 = int(row1["grid_id"])
        mask1 = (labels_t1 == id1)
        y1, x1 = row1["centroid_y"], row1["centroid_x"]

        for _, row2 in df_t2.iterrows():
            id2 = int(row2["grid_id"])
            mask2 = (labels_t2 == id2)
            y2, x2 = row2["centroid_y"], row2["centroid_x"]

            #Overlap
            inter = np.logical_and(mask1, mask2)
            denom = min(mask1.sum(), mask2.sum())
            if denom == 0:
                continue
            overlap_ratio = inter.sum() / denom

            #Distance
            dist = np.hypot(x2 - x1, y2 - y1)

            #Match Criteria
            if overlap_ratio >= overlap_threshold and dist <= dist_threshold:
                matches.append({
                    "t1_id": id1,
                    "t2_id": id2,
                    "overlap": round(overlap_ratio, 3),
                    "distance": round(dist, 2)
                })

    if not matches:
        return pd.DataFrame(columns=["t1_id", "t2_id", "overlap", "distance"])

    return pd.DataFrame(matches) 



##############################
# Classify storm splitting, merging, forming, dissipating
##############################
def classify_relationships(matches, df_t1, df_t2):
    rels = []
    t1_ids = set(df_t1["grid_id"])
    t2_ids = set(df_t2["grid_id"])

    if matches.empty:
        return pd.DataFrame(
            [{"type": "dissipated", "t1_id": did} for did in t1_ids] +
            [{"type": "new", "t2_id": nid} for nid in t2_ids]
        )

    matched_t1 = set(matches["t1_id"])
    matched_t2 = set(matches["t2_id"])

    #Continuations (1 → 1)
    for id1, group in matches.groupby("t1_id"):
        if len(group) == 1:
            rels.append({
                "type": "continue",
                "t1_id": id1,
                "t2_id": group["t2_id"].iloc[0]
            })

    #Splits (1 → many)
    for id1, group in matches.groupby("t1_id"):
        if len(group) > 1:
            rels.append({"type": "split", "t1_id": id1, "t2_ids": group["t2_id"].tolist()})

    #Merges (many → 1)
    for id2, group in matches.groupby("t2_id"):
        if len(group) > 1:
            rels.append({"type": "merge", "t2_id": id2, "t1_ids": group["t1_id"].tolist()})

    #New storms
    new_storms = list(t2_ids - matched_t2)
    for nid in new_storms:
        rels.append({"type": "new", "t2_id": nid})

    #Dissipated storms
    dead_storms = list(t1_ids - matched_t1)
    for did in dead_storms:
        rels.append({"type": "dissipated", "t1_id": did})

    return pd.DataFrame(rels)



##############################
# Update track IDs on matches
##############################
def update_tracks(matches, rels, id_map, next_track_id, parent_map):
    new_map = {}

    #Continuations (1 → 1)
    for _, row in matches.iterrows():
        id1, id2 = int(row["t1_id"]), int(row["t2_id"])
        if id1 in id_map:
            track_id = id_map[id1]
        else:
            track_id = next(next_track_id)
            parent_map[track_id] = track_id
        new_map[id2] = track_id

    #Splits (1 → many)
    for _, row in rels.iterrows():
        if row["type"] == "split":
            parent_tid = id_map.get(row["t1_id"])
            for child_gid in row["t2_ids"]:
                child_tid = next(next_track_id)
                id_map[child_gid] = child_tid
                parent_map[child_tid] = parent_tid

    #Merges (many → 1)
    if row["type"] == "merge":
        merged_tid = next(next_track_id)
        id_map[row["t2_id"]] = merged_tid
        for p in row["t1_ids"]:
            if p in id_map:
                parent_map[merged_tid] = id_map[p]

    #New storms
    handled = set(new_map.keys())
    for nid in map(int, rels.loc[rels["type"] == "new", "t2_id"].tolist()):
        if nid not in handled:
            new_track = next(next_track_id)
            new_map[nid] = new_track
            parent_map[new_track] = new_track

    #Fill in missing parents
    for gid, tid in id_map.items():
        if tid not in parent_map or parent_map[tid] is None:
            parent_map[tid] = tid

    return new_map



##############################
# Summarize storm life-cycle statistics for each tracked storm
##############################
def summarize_tracks(df, parent_map):
    #Frame interval from df order (minutes)
    frame_interval = (
        df["timestamp"].sort_values().diff().dt.total_seconds().dropna().mode()[0] / 60.0
    )

    rows = []
    for tid, g in df.groupby("track_id", sort=False):
        g = g.sort_values("timestamp")
        start = g["timestamp"].iloc[0]
        last  = g["timestamp"].iloc[-1]
        end   = last + pd.Timedelta(minutes=frame_interval) 

        rows.append({
            "storm_id": tid,
            "parent_id": parent_map.get(tid, tid),
            "start_time": start,
            "end_time": end,
            "duration": (end - start).total_seconds() / 60.0,
            "avg_centroid_x": g["centroid_x"].mean(),
            "avg_centroid_y": g["centroid_y"].mean(),
            "avg_dbz": g["peak_dbz"].mean(),
            "avg_area": g["area_px"].mean(),
            "n_frames": g["timestamp"].nunique()
        })

    summary = pd.DataFrame(rows)

    #Lineage Counts
    child_counts = pd.Series(list(parent_map.values())).value_counts()
    summary["num_children"] = summary["storm_id"].map(child_counts).fillna(0).astype(int)
    summary["is_root"] = summary["storm_id"] == summary["parent_id"]

    #Base Class
    summary["classification"] = np.where(summary["is_root"], "root", "continue")
    summary.loc[summary["num_children"] > 1, "classification"] = "split"

    #Dissipated if not reaching the very last frame boundary
    final_boundary = df["timestamp"].max() + pd.Timedelta(minutes=frame_interval)
    died = summary["end_time"] < final_boundary
    summary.loc[died, "classification"] = summary.loc[died, "classification"] + ", dissipated"

    return summary.sort_values(["start_time", "storm_id"]).reset_index(drop=True)

def bytes_to_array(b):
    if b is None:
        return None
    return np.load(io.BytesIO(b), allow_pickle=False)

def simple_hash(values):
    s = ",".join(map(str, values))
    return hashlib.sha256(s.encode()).hexdigest()


##############################
# Main Storm Tracker
##############################
def _root_parent(track_id, parent_map):
    """Return the ultimate root for a track_id."""
    p = parent_map.get(track_id, track_id)
    while p != parent_map.get(p, p):
        p = parent_map[p]
    return p


def track_storms_for_day(df, grids, overlap_threshold=0.02, dist_threshold=20):

    #Prepare Frames
    frames = {ts: g.copy() for ts, g in df.groupby("timestamp", sort=True)}
    timestamps = sorted(frames.keys())


    next_id = count(1)
    parent_map = {} 
    track_records = []
    id_map = {}

    #First Frame
    t0 = timestamps[0]
    df0 = frames[t0]


    for gid in df0["grid_id"]:
        print(gid)
        tid = next(next_id)
        id_map[int(gid)] = tid
        parent_map[tid] = tid


    #Attach First Frame
    df0 = df0.copy()
    df0["track_id"] = df0["grid_id"].map(id_map)
    df0["parent_track_id"] = df0["track_id"].map(lambda z: parent_map.get(z, z))
    track_records.append(df0)


    #Subsequent Frames
    prev_ts, prev_df, prev_labels = t0, df0, grids[t0] 



    for ts in timestamps[1:]:
        # print('path3')
        cur_df = frames[ts].copy()
        cur_labels = grids[ts]

        #Compute matches
        matches = match_frames(prev_labels, prev_df, cur_labels, cur_df,
                               overlap_threshold=overlap_threshold,
                               dist_threshold=dist_threshold,
                               verbose=False)

        # decide relations
        rels = classify_relationships(matches, prev_df, cur_df)

        # Build next_id_map ONLY from this pair of frames (strict)
        next_id_map = {}

        # 1) continuations (1->1)
        cont = rels[rels["type"] == "continue"][["t1_id", "t2_id"]].dropna().astype(int)
        for _, r in cont.iterrows():
            t1, t2 = int(r["t1_id"]), int(r["t2_id"])
            if t1 in id_map:
                next_id_map[t2] = id_map[t1]  # carry same storm id

        # 2) splits (1->many): keep max-overlap child; others new with parent = root
        for _, s in rels[rels["type"] == "split"].iterrows():
            t1 = int(s["t1_id"])
            children = list(map(int, s["t2_ids"]))
            if t1 not in id_map:
                continue
            parent_tid = id_map[t1]
            parent_root = _root_parent(parent_tid, parent_map)

            # choose primary child by max overlap
            m_sub = matches[matches["t1_id"] == t1].set_index("t2_id")
            primary = max(children, key=lambda c: m_sub.loc[c, "overlap"] if c in m_sub.index else -1)

            # primary keeps id
            next_id_map[primary] = parent_tid
            # the rest are brand-new with parent set to root
            for child in children:
                if child == primary:
                    continue
                new_tid = next(next_id)
                next_id_map[child] = new_tid
                parent_map[new_tid] = parent_root

        # 3) merges (many->1): winner by max overlap keeps id; others end
        for _, m in rels[rels["type"] == "merge"].iterrows():
            t2 = int(m["t2_id"])
            parents = list(map(int, m["t1_ids"]))
            # choose parent with max overlap
            m_sub = matches[matches["t2_id"] == t2].set_index("t1_id")
            winner = max(parents, key=lambda p: m_sub.loc[p, "overlap"] if p in m_sub.index else -1)
            if winner in id_map:
                next_id_map[t2] = id_map[winner]
                # ensure lineage preserved
                wid = id_map[winner]
                parent_map[wid] = parent_map.get(wid, wid)

        # 4) brand-new storms at t (no inbound match)
        new_ids = set(cur_df["grid_id"].astype(int)) - set(next_id_map.keys())
        for nid in sorted(new_ids):
            new_tid = next(next_id)
            next_id_map[nid] = new_tid
            parent_map[new_tid] = new_tid  #new root (no continuity from t-Δt)

        #Replace current mapping with newly decided mapping
        id_map = next_id_map

        #Attach current frame rows with chosen ids
        cur_df["track_id"] = cur_df["grid_id"].map(id_map)
        cur_df["parent_track_id"] = cur_df["track_id"].map(lambda z: parent_map.get(z, z))
        track_records.append(cur_df)

        prev_ts, prev_df, prev_labels = ts, cur_df, cur_labels

    #Aggregate all frames
    full_tracks = pd.concat(track_records, ignore_index=True).sort_values(["timestamp", "track_id"])

    #Summarize
    summary = summarize_tracks(full_tracks, parent_map)

    #Rebase storm ids to 1..N and keep parent linkage consistent
    rebase = {old: i for i, old in enumerate(sorted(summary["storm_id"].unique()), start=1)}
    summary["storm_id"] = summary["storm_id"].map(rebase)
    summary["parent_id"] = summary["parent_id"].map(lambda x: rebase.get(x, rebase.get(x, x))).astype(int)
    full_tracks["track_id"] = full_tracks["track_id"].map(rebase)
    full_tracks["parent_track_id"] = full_tracks["track_id"].map(lambda x: summary.set_index("storm_id").loc[x, "parent_id"])

    #Per-storm, Per-frame lists
    per_storm_lists = (
        full_tracks.sort_values(["track_id", "timestamp"])
        .groupby("track_id", sort=False)
        .apply(lambda g: pd.Series({
            # "obs_id_list":  ",".join(map(str, g["obs_id"].tolist())),
            # "anchor_x_list": ",".join(f"{v:.2f}" for v in g["anchor_x"].tolist()),
            # "anchor_y_list": ",".join(f"{v:.2f}" for v in g["anchor_y"].tolist()),
            # "area_list":     ",".join(f"{v:.3f}" for v in g["area_px"].tolist()),
            # "n_list_frames": len(g)  # quick check
            "obs_id_list":  g["obs_id"].astype(int).tolist(),
            "anchor_x_list": g["anchor_x"].round(2).tolist(),
            "anchor_y_list": g["anchor_y"].round(2).tolist(),
            "area_list":     g["area_px"].round(3).tolist(),
            "n_list_frames": len(g)  # quick check
        }))
        .reset_index()
        .rename(columns={"track_id": "storm_id"})
    )

    per_storm_lists["obs_id_hash"] = per_storm_lists["obs_id_list"].apply(simple_hash)

    summary = summary.merge(per_storm_lists, on="storm_id", how="left")
    
    summary = summary.rename(columns={"storm_id": "track_id"})

    #Rounding of values
    summary = summary.round({
        "avg_centroid_x": 2, "avg_centroid_y": 2, "avg_dbz": 2, "avg_area": 3
    })

    summary = summary[
        [
            "track_id", "parent_id", "start_time", "end_time", "duration",
            "avg_centroid_x", "avg_centroid_y", "avg_dbz", "avg_area",
            "n_frames", "num_children", "classification",
            "obs_id_list", "anchor_x_list", "anchor_y_list", "area_list",'obs_id_hash'
        ]
    ]
    return summary, full_tracks, parent_map

In [47]:
# Load environment variables from .env file
load_dotenv()

# Read credentials
DB_HOST = os.getenv("DB_HOST")
DB_PORT = int(os.getenv("DB_PORT", 5432))
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")

# Create connection
conn = pg8000.connect(
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
    user=DB_USER,
    password=DB_PASS
)

print("✅ Connected successfully to PostgreSQL!")

✅ Connected successfully to PostgreSQL!


# main backfil

In [48]:
conn.rollback()

In [43]:
##################################################################################################

# === inputs ===
start_date_str = "2025-10-09"
end_date_str   = "2025-10-09"

# === setup ===
start_date = datetime.strptime(start_date_str, "%Y-%m-%d").date()
end_date   = datetime.strptime(end_date_str, "%Y-%m-%d").date()


current_date = start_date

# === main loop ===
while current_date <= end_date:
    # generate all hours in the day
    start_ts = datetime.combine(current_date, time(0, 0))       # 00:00:00
    end_ts   = datetime.combine(current_date, time(23, 55))     # 23:55:00


    # batch processing 
    print(f'processing for {start_ts} and {end_ts}')

    storm_obs_query = """
    SELECT * 
    FROM storm_observation 
    WHERE 
        timestamp between %s and %s;
    """
    
    storm_obs = query_to_df(conn,storm_obs_query, (start_ts,end_ts))
    
    storm_grid_query = """
    SELECT * 
    FROM storm_grid 
    WHERE 
        timestamp between %s and %s;
    """

    storm_grid = query_to_df(conn,storm_grid_query, (start_ts,end_ts))
    storm_grid["grid_data"] = storm_grid["grid_data"].apply(bytes_to_array)
    storm_grid_dict = storm_grid.set_index("timestamp")["grid_data"].to_dict()


    if storm_obs.shape[0] == 0 or storm_grid.shape[0] == 0: 
        continue
    summary ,_,_  = track_storms_for_day(storm_obs,storm_grid_dict)
    summary_df = summary.loc[summary['duration']>0].copy()

    # none handling
    summary_df = summary_df.replace({np.nan: None})

    # sql query
    insert_sql = """
    INSERT INTO storm (
        track_id, parent_id, start_time, end_time, duration,
        avg_centroid_x, avg_centroid_y, avg_dbz, avg_area,
        n_frames, num_children, classification,
        obs_id_list, anchor_x_list, anchor_y_list, area_list,obs_id_hash
    )
    VALUES (
        %s, %s, %s, %s, %s,
        %s, %s, %s, %s,
        %s, %s, %s,
        %s, %s, %s, %s ,%s
    )
    ON CONFLICT (obs_id_hash) DO UPDATE
    SET
        track_id        = EXCLUDED.track_id,
        parent_id       = EXCLUDED.parent_id,
        start_time      = EXCLUDED.start_time,
        end_time        = EXCLUDED.end_time,
        duration        = EXCLUDED.duration,
        avg_centroid_x  = EXCLUDED.avg_centroid_x,
        avg_centroid_y  = EXCLUDED.avg_centroid_y,
        avg_dbz         = EXCLUDED.avg_dbz,
        avg_area        = EXCLUDED.avg_area,
        n_frames        = EXCLUDED.n_frames,
        num_children    = EXCLUDED.num_children,
        classification  = EXCLUDED.classification,
        obs_id_list     = EXCLUDED.obs_id_list,
        anchor_x_list   = EXCLUDED.anchor_x_list,
        anchor_y_list   = EXCLUDED.anchor_y_list,
        area_list       = EXCLUDED.area_list;
    """

    # convert df
    rows = list(
        summary_df[[
            "track_id", "parent_id", "start_time", "end_time", "duration",
            "avg_centroid_x", "avg_centroid_y", "avg_dbz", "avg_area",
            "n_frames", "num_children", "classification",
            "obs_id_list", "anchor_x_list", "anchor_y_list", "area_list",'obs_id_hash'
        ]].itertuples(index=False, name=None)
    )

    # insert db
    try:
        with conn.cursor() as cur:
            cur.executemany(insert_sql, rows)
        conn.commit()
        print(f" Inserted or updated {len(rows)} rows into 'storm'.")
    except Exception as e:
        conn.rollback()
        print(" Database error while inserting into storm:", e)
        raise
        
    current_date += timedelta(days=1)




processing for 2025-10-09 00:00:00 and 2025-10-09 23:55:00
2
3


/var/folders/c9/rl6pcdvd1_xb6ng3n8l9wr2w0000gn/T/ipykernel_29158/4049882603.py:397: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


 Inserted or updated 9 rows into 'storm'.


#### markdown

In [33]:
# ##################################################################################################

# # === inputs ===
# start_date_str = "2025-10-09"
# end_date_str   = "2025-10-09"

# # === setup ===
# start_date = datetime.strptime(start_date_str, "%Y-%m-%d").date()
# end_date   = datetime.strptime(end_date_str, "%Y-%m-%d").date()


# current_date = start_date

# # === main loop ===
# while current_date <= end_date:
#     # generate all hours in the day
#     hours = [datetime.combine(current_date, time(h, 0)) for h in range(24)]

#     for i in range(len(hours)):
#         start_ts = hours[i]
#         # for the last hour, end at 23:55 instead of 00:00 next day
#         if i == len(hours) - 1:
#             end_ts = datetime.combine(current_date, time(23, 55))
#         else:
#             end_ts = hours[i] + timedelta(hours=1)

#         # batch processing 
#         print(f'processing for {start_ts} and {end_ts}')

#         storm_obs_query = """
#         SELECT * 
#         FROM storm_observation 
#         WHERE 
#             timestamp between %s and %s;
#         """
        
#         storm_obs = query_to_df(conn,storm_obs_query, (start_ts,end_ts))
        
#         storm_grid_query = """
#         SELECT * 
#         FROM storm_grid 
#         WHERE 
#             timestamp between %s and %s;
#         """

#         storm_grid = query_to_df(conn,storm_grid_query, (start_ts,end_ts))
#         storm_grid["grid_data"] = storm_grid["grid_data"].apply(bytes_to_array)
#         storm_grid_dict = storm_grid.set_index("timestamp")["grid_data"].to_dict()


#         if storm_obs.shape[0] == 0 or storm_grid.shape[0] == 0: 
#             continue
#         summary ,_,_  = track_storms_for_day(storm_obs,storm_grid_dict)
#         summary_df = summary.loc[summary['duration']>0].copy()

#         # none handling
#         summary_df = summary_df.replace({np.nan: None})

#         # sql query
#         insert_sql = """
#         INSERT INTO dummy_storm (
#             track_id, parent_id, start_time, end_time, duration,
#             avg_centroid_x, avg_centroid_y, avg_dbz, avg_area,
#             n_frames, num_children, classification,
#             obs_id_list, anchor_x_list, anchor_y_list, area_list,obs_id_hash
#         )
#         VALUES (
#             %s, %s, %s, %s, %s,
#             %s, %s, %s, %s,
#             %s, %s, %s,
#             %s, %s, %s, %s ,%s
#         )
#         ON CONFLICT (obs_id_hash) DO UPDATE
#         SET
#             track_id        = EXCLUDED.track_id,
#             parent_id       = EXCLUDED.parent_id,
#             start_time      = EXCLUDED.start_time,
#             end_time        = EXCLUDED.end_time,
#             duration        = EXCLUDED.duration,
#             avg_centroid_x  = EXCLUDED.avg_centroid_x,
#             avg_centroid_y  = EXCLUDED.avg_centroid_y,
#             avg_dbz         = EXCLUDED.avg_dbz,
#             avg_area        = EXCLUDED.avg_area,
#             n_frames        = EXCLUDED.n_frames,
#             num_children    = EXCLUDED.num_children,
#             classification  = EXCLUDED.classification,
#             obs_id_list     = EXCLUDED.obs_id_list,
#             anchor_x_list   = EXCLUDED.anchor_x_list,
#             anchor_y_list   = EXCLUDED.anchor_y_list,
#             area_list       = EXCLUDED.area_list;
#         """

#         # convert df
#         rows = list(
#             summary_df[[
#                 "track_id", "parent_id", "start_time", "end_time", "duration",
#                 "avg_centroid_x", "avg_centroid_y", "avg_dbz", "avg_area",
#                 "n_frames", "num_children", "classification",
#                 "obs_id_list", "anchor_x_list", "anchor_y_list", "area_list",'obs_id_hash'
#             ]].itertuples(index=False, name=None)
#         )

#         # insert db
#         try:
#             with conn.cursor() as cur:
#                 cur.executemany(insert_sql, rows)
#             conn.commit()
#             print(f" Inserted or updated {len(rows)} rows into 'storm'.")
#         except Exception as e:
#             conn.rollback()
#             print(" Database error while inserting into storm:", e)
#             raise
            
#     current_date += timedelta(days=1)


processing for 2025-10-09 00:00:00 and 2025-10-09 01:00:00
processing for 2025-10-09 01:00:00 and 2025-10-09 02:00:00
processing for 2025-10-09 02:00:00 and 2025-10-09 03:00:00
processing for 2025-10-09 03:00:00 and 2025-10-09 04:00:00
processing for 2025-10-09 04:00:00 and 2025-10-09 05:00:00
processing for 2025-10-09 05:00:00 and 2025-10-09 06:00:00
processing for 2025-10-09 06:00:00 and 2025-10-09 07:00:00
processing for 2025-10-09 07:00:00 and 2025-10-09 08:00:00
processing for 2025-10-09 08:00:00 and 2025-10-09 09:00:00
processing for 2025-10-09 09:00:00 and 2025-10-09 10:00:00
2
3


/var/folders/c9/rl6pcdvd1_xb6ng3n8l9wr2w0000gn/T/ipykernel_29158/4049882603.py:397: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


 Inserted or updated 3 rows into 'storm'.
processing for 2025-10-09 10:00:00 and 2025-10-09 11:00:00
3
4
5


/var/folders/c9/rl6pcdvd1_xb6ng3n8l9wr2w0000gn/T/ipykernel_29158/4049882603.py:397: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


 Inserted or updated 7 rows into 'storm'.
processing for 2025-10-09 11:00:00 and 2025-10-09 12:00:00
2


/var/folders/c9/rl6pcdvd1_xb6ng3n8l9wr2w0000gn/T/ipykernel_29158/4049882603.py:397: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


 Inserted or updated 1 rows into 'storm'.
processing for 2025-10-09 12:00:00 and 2025-10-09 13:00:00
processing for 2025-10-09 13:00:00 and 2025-10-09 14:00:00
processing for 2025-10-09 14:00:00 and 2025-10-09 15:00:00
processing for 2025-10-09 15:00:00 and 2025-10-09 16:00:00
processing for 2025-10-09 16:00:00 and 2025-10-09 17:00:00
processing for 2025-10-09 17:00:00 and 2025-10-09 18:00:00
processing for 2025-10-09 18:00:00 and 2025-10-09 19:00:00
processing for 2025-10-09 19:00:00 and 2025-10-09 20:00:00
processing for 2025-10-09 20:00:00 and 2025-10-09 21:00:00
processing for 2025-10-09 21:00:00 and 2025-10-09 22:00:00
processing for 2025-10-09 22:00:00 and 2025-10-09 23:00:00
processing for 2025-10-09 23:00:00 and 2025-10-09 23:55:00


In [45]:
q = """
SELECT * 
FROM dummy_storm
"""

test = query_to_df(conn,q)


In [35]:
test = test.sort_values(by="start_time", ascending=True)
test

,storm_id,track_id,parent_id,start_time,end_time,duration,avg_centroid_x,avg_centroid_y,avg_dbz,avg_area,n_frames,num_children,classification,obs_id_list,anchor_x_list,anchor_y_list,area_list,obs_id_hash
0,1,1,1,2025-10-09 09:20:00,2025-10-09 09:50:00,30.0,26.51,37.35,69.83,199.921,6,1,"root, dissipated","[1, 3, 5, 7, 10, 13]","[4.0, 18.0, 24.0, 26.0, 38.0, 48.0]","[31.0, 39.0, 38.0, 36.0, 39.0, 40.0]","[11.0, 200.992, 187.088, 210.496, 269.984, 319...",e1bc6478d546d8e55deeb12007560c7cb5b0e5e17ab000...
1,2,3,3,2025-10-09 09:30:00,2025-10-09 10:00:00,30.0,25.37,100.20,67.00,20.768,7,1,root,"[4, 6, 9, 12, 14, 16, 19]","[9.0, 16.0, 23.0, 30.0, 28.0, 33.0, 38.0]","[100.0, 100.0, 103.0, 100.0, 99.0, 100.0, 99.0]","[11.616, 20.768, 34.76, 24.288, 19.184, 18.128...",50f25461059be422fb407758c87cb7819d424af3ba2782...
2,3,6,6,2025-10-09 09:55:00,2025-10-09 10:00:00,5.0,71.80,84.32,65.00,9.240,2,1,root,"[15, 17]","[70.0, 74.0]","[85.0, 84.0]","[8.536, 9.944]",35f5113aa6d923af58f54554eb326c5a8030ab2fe9266e...
3,4,3,3,2025-10-09 10:00:00,2025-10-09 10:05:00,5.0,46.62,98.28,65.00,12.540,2,2,"split, dissipated","[19, 22]","[38.0, 55.0]","[99.0, 97.0]","[16.632, 8.448]",cd1b25c12b2de7832af2739c9a60e67ebb29ef8ed01538...
4,5,4,3,2025-10-09 10:05:00,2025-10-09 10:20:00,15.0,21.13,100.68,64.00,29.128,4,0,"continue, dissipated","[21, 26, 29, 32]","[19.0, 18.0, 24.0, 24.0]","[103.0, 101.0, 99.0, 99.0]","[29.216, 35.728, 37.048, 14.52]",e4c9ad9e7e0fea18d0ffd23df6bbe2a3482c644874df6d...
5,6,5,5,2025-10-09 10:05:00,2025-10-09 10:15:00,10.0,81.69,35.89,69.00,248.864,3,1,"root, dissipated","[20, 23, 28]","[74.0, 85.0, 86.0]","[34.0, 37.0, 37.0]","[312.488, 243.496, 190.608]",ce20d3b26f00ef42bcc5a346474abb7aefd3d57443c130...
6,7,8,8,2025-10-09 10:10:00,2025-10-09 10:15:00,5.0,110.38,99.54,68.50,7.172,2,1,"root, dissipated","[27, 30]","[107.0, 114.0]","[98.0, 101.0]","[7.304, 7.04]",5adf9ea538c99befa95270dc9a1abe86a7fbc5125f9123...
7,8,9,9,2025-10-09 10:20:00,2025-10-09 10:25:00,5.0,79.65,9.78,65.00,21.912,2,1,"root, dissipated","[31, 33]","[78.0, 81.0]","[13.0, 7.0]","[31.064, 12.76]",aeeddc78473cd1f37381170af6d235d2b8d94a014a710a...
8,9,11,11,2025-10-09 10:25:00,2025-10-09 10:40:00,15.0,169.04,54.76,69.00,26.268,4,1,"root, dissipated","[35, 36, 37, 39]","[157.0, 165.0, 174.0, 180.0]","[56.0, 56.0, 56.0, 50.0]","[14.696, 21.648, 29.656, 39.072]",e802d91d92103fd2be513518464f24a4ba845e1c39e7fe...
9,10,13,13,2025-10-09 10:45:00,2025-10-09 11:00:00,15.0,23.23,99.62,66.50,44.836,4,1,root,"[40, 42, 44, 45]","[16.0, 22.0, 25.0, 30.0]","[104.0, 101.0, 98.0, 95.0]","[51.216, 46.992, 44.616, 36.52]",e57732e91c1d23cae2d288be3934817878af8e5fd560e6...


In [46]:
test = test.sort_values(by="start_time", ascending=True)
test

,storm_id,track_id,parent_id,start_time,end_time,duration,avg_centroid_x,avg_centroid_y,avg_dbz,avg_area,n_frames,num_children,classification,obs_id_list,anchor_x_list,anchor_y_list,area_list,obs_id_hash
0,1,1,1,2025-10-09 09:20:00,2025-10-09 09:50:00,30.0,26.51,37.35,69.83,199.921,6,1,"root, dissipated","[1, 3, 5, 7, 10, 13]","[4.0, 18.0, 24.0, 26.0, 38.0, 48.0]","[31.0, 39.0, 38.0, 36.0, 39.0, 40.0]","[11.0, 200.992, 187.088, 210.496, 269.984, 319...",e1bc6478d546d8e55deeb12007560c7cb5b0e5e17ab000...
1,2,3,3,2025-10-09 09:30:00,2025-10-09 10:05:00,35.0,29.07,99.82,66.75,19.228,8,2,"split, dissipated","[4, 6, 9, 12, 14, 16, 19, 22]","[9.0, 16.0, 23.0, 30.0, 28.0, 33.0, 38.0, 55.0]","[100.0, 100.0, 103.0, 100.0, 99.0, 100.0, 99.0...","[11.616, 20.768, 34.76, 24.288, 19.184, 18.128...",15290dfb069a722b78bebdee5132f190088f82f713ee3b...
2,3,6,6,2025-10-09 09:55:00,2025-10-09 10:00:00,5.0,71.80,84.32,65.00,9.240,2,1,"root, dissipated","[15, 17]","[70.0, 74.0]","[85.0, 84.0]","[8.536, 9.944]",35f5113aa6d923af58f54554eb326c5a8030ab2fe9266e...
3,4,8,3,2025-10-09 10:05:00,2025-10-09 10:20:00,15.0,21.13,100.68,64.00,29.128,4,0,"continue, dissipated","[21, 26, 29, 32]","[19.0, 18.0, 24.0, 24.0]","[103.0, 101.0, 99.0, 99.0]","[29.216, 35.728, 37.048, 14.52]",e4c9ad9e7e0fea18d0ffd23df6bbe2a3482c644874df6d...
4,5,9,9,2025-10-09 10:05:00,2025-10-09 10:15:00,10.0,81.69,35.89,69.00,248.864,3,1,"root, dissipated","[20, 23, 28]","[74.0, 85.0, 86.0]","[34.0, 37.0, 37.0]","[312.488, 243.496, 190.608]",ce20d3b26f00ef42bcc5a346474abb7aefd3d57443c130...
5,6,12,12,2025-10-09 10:10:00,2025-10-09 10:15:00,5.0,110.38,99.54,68.50,7.172,2,1,"root, dissipated","[27, 30]","[107.0, 114.0]","[98.0, 101.0]","[7.304, 7.04]",5adf9ea538c99befa95270dc9a1abe86a7fbc5125f9123...
6,7,13,13,2025-10-09 10:20:00,2025-10-09 10:25:00,5.0,79.65,9.78,65.00,21.912,2,1,"root, dissipated","[31, 33]","[78.0, 81.0]","[13.0, 7.0]","[31.064, 12.76]",aeeddc78473cd1f37381170af6d235d2b8d94a014a710a...
7,8,15,15,2025-10-09 10:25:00,2025-10-09 10:40:00,15.0,169.04,54.76,69.00,26.268,4,1,"root, dissipated","[35, 36, 37, 39]","[157.0, 165.0, 174.0, 180.0]","[56.0, 56.0, 56.0, 50.0]","[14.696, 21.648, 29.656, 39.072]",e802d91d92103fd2be513518464f24a4ba845e1c39e7fe...
8,9,17,17,2025-10-09 10:45:00,2025-10-09 11:10:00,25.0,29.03,97.77,64.83,35.787,6,1,root,"[40, 42, 44, 45, 46, 47]","[16.0, 22.0, 25.0, 30.0, 39.0, 42.0]","[104.0, 101.0, 98.0, 95.0, 95.0, 93.0]","[51.216, 46.992, 44.616, 36.52, 22.968, 12.408]",f9533be250a36ba3a606248070b3c6900476479cec74e3...
